In [1]:
import pennylane as qml
import numpy as np

# import pyqsp
# from pyqsp.angle_sequence import QuantumSignalProcessingPhases
from numpy.polynomial import Chebyshev, Polynomial

In [2]:
# ==========================================
# PARTE 1: I DATI E I REGISTRI/FILI
# ==========================================
# 1. Partiamo da una VERA matrice diagonale 4x4 (L'input)
# X_matrice = np.array([
#     [ 0.5,   0,   0,   0],
#     [  0, -0.2,   0,   0],
#     [  0,   0, 0.8,   0],
#     [  0,   0,   0,   0.1]
# ])

X_matrice = np.array([
    [ 0.7,   0,   0,   0],
    [  0, -4,   0,   0],
    [  0,   0, 8,   0],
    [  0,   0,   0,   0.3]
])

# X_matrice = np.array([
#     [ -720,   0,   0,   0],
#     [  0, 56,   0,   0],
#     [  0,   0, 200,   0],
#     [  0,   0,   0,   170]
# ])

# X_matrice = np.array([
#     [ -720,   0],
#     [  0, 56]
# ])

# X_matrice = np.array([
#     [ -720,   0,   0],
#     [  0, 56,   0],
#     [  0,   0, 200],
# ])

print("1. La matrice X di partenza è:")
print(X_matrice)
print("\n")


# 2. Estraiamo la diagonale classica
x_classico_grezzo = np.diag(X_matrice)
N_originale = len(x_classico_grezzo)

print("2. La diagonale di X e la sua dimensione:")
print("x_classico_grezzo =", x_classico_grezzo) 
print("N =", N_originale)
print("\n")

# 3. Calcolo qubit e Padding
# Calcoliamo n: il numero di qubit necessari (logaritmo in base 2 arrotondato per eccesso)
# Es: N=2 -> n=1. N=3 -> n=2. N=4 -> n=2.
n = int(np.ceil(np.log2(N_originale)))

# Calcoliamo quanti slot fisici avrà la memoria quantistica (la potenza di 2 più vicina)
N_pad = 2**n

# Facciamo il padding: aggiungiamo zeri alla fine se N_originale è minore di N_pad
# (Fondamentale per le matrici 3x3, 5x5, ecc.)
x_classico = np.pad(x_classico_grezzo, (0, N_pad - N_originale))

print("3. Calcolo qubit e Padding")
print("n =", n)
print("N_pad =", n)
print("x_classico =", x_classico)
print("\n")

# 4. Normalizzazione del vettore (obbligatoria per le ampiezze, sennò il computer quantistico non può leggere i dati)
norma_originale = np.linalg.norm(x_classico)
x = x_classico / norma_originale 

print("4. Normalizzazione di x_classico")
print("norma_originale =", norma_originale)
print("x =", x)

# 5. Assegniamo i fili. n=2 (perché abbiamo 4 dati, 2^2=4)
# Servono n fili per address, n per data, 1 per l'ancilla B, 1 per l'ancilla lcu.
# Generiamo le liste dei fili in base alla 'n' calcolata!
ancilla_lcu = [0]                                    # Il nuovo qubit (+1) in alto
qubits_address = list(range(1, n+1))                   # Es: se n=1 -> [0]
qubits_dati = list(range(n+1, 2*n +1))                    # Es: se n=1 -> [1]
ancilla_b = [2*n +1]                                    # Es: se n=1 -> [2]

# Prepariamo un raggruppamento che ci servirà per S_0
qubits_da_B = qubits_dati + ancilla_b

# Creiamo una lista unica con tutti i 5 fili nell'ordine corretto
tutti_i_fili = ancilla_lcu + qubits_address + qubits_dati + ancilla_b

1. La matrice X di partenza è:
[[ 0.7  0.   0.   0. ]
 [ 0.  -4.   0.   0. ]
 [ 0.   0.   8.   0. ]
 [ 0.   0.   0.   0.3]]


2. La diagonale di X e la sua dimensione:
x_classico_grezzo = [ 0.7 -4.   8.   0.3]
N = 4


3. Calcolo qubit e Padding
n = 2
N_pad = 2
x_classico = [ 0.7 -4.   8.   0.3]


4. Normalizzazione di x_classico
norma_originale = 8.976636341080104
x = [ 0.07798021 -0.44560121  0.89120242  0.03342009]


In [3]:
# ==========================================
# PARTE 2: ORACOLO E RIFLESSIONE
# ==========================================

def U_x(wires):
    """
    L'Oracolo: Inietta il vettore x nelle ampiezze dei qubit bersaglio.
    """
     #qml.AmplitudeEmbedding(features=x, wires=wires, normalize=True)
    qml.MottonenStatePreparation(x, wires=wires)

def S_0():
    """
    La Riflessione: La matrice diagonale con un -1 in alto a sinistra.
    Inverte il segno SOLO se i qubit 'dati' e 'B' sono tutti a zero.
    Non tocca gli 'address' per non rovinare le coordinate!
    """
    stato_zero = np.zeros(len(qubits_da_B), dtype=int)
    qml.FlipSign(stato_zero, wires=qubits_da_B)



# ==========================================
# PARTE 3: W (dalla Figura 1)
# ==========================================

def W():
    """
    Allinea i dati x_k con le coordinate corrette della matrice diagonale.
    """
    # 1. U normale solo sui dati
    U_x(wires=qubits_dati)
    
    # 2. Hadamard sull'ancilla B
    qml.Hadamard(wires=ancilla_b)
    
    # 3. U inverso sui dati, controllato da B
    # qml.adjoint ribalta l'oracolo, qml.ctrl lo subordina a B
    qml.ctrl(qml.adjoint(U_x), control=ancilla_b)(wires=qubits_dati)
    
    # 4. Le porte Toffoli incrociate (da address a dati, controllate da B)
    # Servono a "saldare" il dato corretto all'indirizzo corretto
    for i in range(len(qubits_address)):
        # Toffoli prende: [controllo_1, controllo_2, bersaglio]
        qml.Toffoli(wires=[ancilla_b[0], qubits_address[i], qubits_dati[i]])
        
    # 5. Hadamard finale sull'ancilla B
    qml.Hadamard(wires=ancilla_b)



# ==========================================
# PARTE 4: G
# ==========================================

def G():
    # Costruisce l'operatore G = W S_0 W^\dagger Z_B.
    # Le operazioni si scrivono nell'ordine esatto in cui toccano i qubit.
    
    # 1. Z_B (Riflessione sull'ancilla)
    qml.PauliZ(wires=ancilla_b)
    
    # 2. W^\dagger (Smontaggio: PennyLane inverte tutto W in automatico)
    qml.adjoint(W)()
    
    # 3. S_0 (Riflessione sull'origine)
    S_0()
    
    # 4. W (Rimontaggio)
    W()



# ==========================================
# PARTE 5: Gtilde
# ==========================================

def G_tilde():
    """Implementa -1/2 (G + G_dagger) usando le porte esatte del paper"""
    
    # 1. Hadamard iniziale sull'ancilla LCU
    qml.Hadamard(wires=ancilla_lcu)
    
    # 2. Sandwich PauliX per il controllo su 0 (Pallino bianco)
    qml.PauliX(wires=ancilla_lcu)
    qml.ctrl(G, control=ancilla_lcu)()  # G controllato standard
    qml.PauliX(wires=ancilla_lcu)
    
    # 3. Controllo su 1 per G_dagger (Pallino nero)
    qml.ctrl(qml.adjoint(G), control=ancilla_lcu)()
    
    # 4. Hadamard finale per ricongiungere gli stati
    qml.Hadamard(wires=ancilla_lcu)
    
    # 5. La rotazione Rz(2pi) per ottenere il segno MENO globale
    qml.RZ(2 * np.pi, wires=ancilla_lcu)


In [4]:
# ==========================================
# PARTE 6: QSVT (Il Programma Non Lineare)
# ==========================================

# --- GENERATORE UNIVERSALE ANGOLI ---
def genera_angoli_qsvt(funzione=None, coeff_diretti=None, grado_polinomio=3, tipo_parita='dispari'):
    """Calcola gli angoli di fase partendo da una f(x) o da un polinomio esatto"""
    print("\n--- GENERATORE DI ANGOLI QSVT ---")
    coefficienti = None
    
    # OPZIONE A: Polinomio diretto
    if coeff_diretti is not None:
        print("Modalità: Inserimento diretto del Polinomio")
        coefficienti = np.array(coeff_diretti, dtype=float)
        
    # OPZIONE B: Funzione generica f(x) (Approssimazione di Chebyshev)
    elif funzione is not None:
        print(f"Modalità: Approssimazione di Chebyshev di f(x) (Grado {grado_polinomio})")
        x_punti = np.linspace(-1, 1, 1000)
        y_punti = funzione(x_punti)
        cheb_approssimazione = Chebyshev.fit(x_punti, y_punti, deg=grado_polinomio)
        pol_standard = cheb_approssimazione.convert(kind=Polynomial)
        coefficienti = pol_standard.coef.copy()
        
    else:
        raise ValueError("Devi fornire una funzione o dei coefficienti!")

    # Pulizia Parità (obbligatoria per QSVT standard)
    for i in range(len(coefficienti)):
        if tipo_parita == 'dispari' and i % 2 == 0:
            coefficienti[i] = 0.0
        elif tipo_parita == 'pari' and i % 2 != 0:
            coefficienti[i] = 0.0

    # TAGLIO DEGLI ZERI (RISOLVE L'ERRORE DEL 6° GRADO!) <---
    # Rimuove gli zeri creati dal pulitore in coda all'array, così PyQSP non va in errore.
    coefficienti = np.trim_zeros(coefficienti, 'b')
    if len(coefficienti) == 0:
        coefficienti = np.array([0.0])

    # Controllo e normalizzazione del limite quantistico [-1, 1]
    x_test = np.linspace(-1, 1, 1000)
    max_valore = np.max(np.abs(np.polyval(coefficienti[::-1], x_test)))
    
    # SALVIAMO IL FATTORE DI SCALA <---
    fattore_di_scala = 1.0
    if max_valore > 0.0:
        fattore_di_scala = max_valore / 0.999
        # Rimpiccioliamo il polinomio e salviamo di quanto lo abbiamo ridotto
        coefficienti = coefficienti / fattore_di_scala
        
    # # Calcolo angoli PyQSP DEPRECATO DA ME STESSO
    # angoli = np.array(QuantumSignalProcessingPhases(coefficienti, signal_operator="Wx"))

    test_poly_to_angles(coefficienti)
    
    # Calcolo angoli
    angoli = np.array(qml.poly_to_angles(coefficienti, "QSVT", angle_solver='root-finding'))
        
    # RESTITUIAMO SIA GLI ANGOLI CHE LA SCALA <---
    return angoli, fattore_di_scala

# --- TEST POLY_TO_ANGLES ---
def test_poly_to_angles(poly):
    qsvt_angles = qml.poly_to_angles(poly, "QSVT")
    
    t = 0.2
    
    # Encode x in the top left of the matrix
    block_encoding = qml.RX(2 * np.arccos(t), wires=0)
    projectors = [qml.PCPhase(angle, dim=1, wires=0) for angle in qsvt_angles]
    
    @qml.qnode(qml.device("default.qubit"))
    def circuit_qsvt():
        qml.QSVT(block_encoding, projectors)
        return qml.state()
    
    output = qml.matrix(circuit_qsvt, wire_order=[0])()[0, 0]
    expected = sum(coef * (t**i) for i, coef in enumerate(poly))
    
    print("output qsvt: ", output.real)
    print("P(t) =       ", expected)

    
# --- STAMPA POLINOMIO ---
def stampa_polinomio(coeff_ascendenti):
    """
    Prende una lista di coefficienti [x^0, x^1, x^2...] 
    e restituisce la stringa formattata (es: 0.1x^3 + 0.2x)
    """
    termini = []
    grado_max = len(coeff_ascendenti) - 1
    
    # Scorriamo la lista al contrario (dal grado più alto al più basso)
    for i, c in enumerate(reversed(coeff_ascendenti)):
        esponente = grado_max - i
        
        # Saltiamo i coefficienti a zero
        if c == 0:
            continue
            
        # Formattiamo in base all'esponente
        if esponente == 0:
            termini.append(f"{c}")
        elif esponente == 1:
            termini.append(f"{c}x")
        else:
            termini.append(f"{c}x^{esponente}")
            
    # Uniamo i pezzi e sistemiamo i segni negativi (evita "+ -")
    se_vuoto = "0" if not termini else " + ".join(termini).replace("+ -", "- ")
    return se_vuoto

# 1. Definiamo la nostra funzione o il nostro polinomio
print(f"\n--- IMPOSTAZIONE QSVT ---")

# --- SCEGLI QUI QUALE METODO USARE ---

# METODO A: Polinomio Diretto (Attivo)
# Inserisci qui i coefficienti dal grado PIÙ BASSO (x^0) al PIÙ ALTO.
# Puoi aggiungere quanti numeri vuoi!
# Esempio: 0.1x^3 + 0.4x^2 + 0.2x + 0.3

# lista_coefficienti = [0.2, 0.2, 1.6, 0.3]
# lista_coefficienti = [0, 0, 1, 0]
# lista_coefficienti = [0.3, 0.2, 0.4, 0.1]
# lista_coefficienti = [0.2, 0.2, 0.3, 0.1, 0.1, 0.05] 
lista_coefficienti = [0.2, 0.2, 0.3, 0.1, 0.1, 0.05, 0.05]
# lista_coefficienti = [0, 1, 0, 0]
# lista_coefficienti = [0.1, 0.4, 0.2, 0.3]
# lista_coefficienti = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.4]
# lista_coefficienti = [0, 0, 0, 1]

# Se volessi fare un polinomio di 5° grado ti basterebbe allungarla:
# lista_coefficienti = [0.5, 0, -1.2, 0, 0, 0.8] # 0.8x^5 - 1.2x^2 + 0.5

print(f"Vogliamo applicare il polinomio: P(x) = {stampa_polinomio(lista_coefficienti)}")


# METODO B: Funzione Generica (Decommenta per usare questo!)
# def mia_funzione(x):
#     return np.sin(x)
# print("Vogliamo applicare la funzione: f(x) = sin(x)")

# 2. La Traduzione in Angoli di Fase (Phase Angles)
# In un ambiente di produzione reale, qui chiameresti un "angle solver" classico.
try:
    # Generiamo gli angoli veri con PyQSP!
    # Se usi il METODO B, cambia i parametri qui sotto in: funzione=mia_funzione
    # ---> RICEVIAMO IL FATTORE DI SCALA 'scala_qsvt' <---
    angoli_fase_qsvt, scala_qsvt = genera_angoli_qsvt(coeff_diretti=lista_coefficienti, tipo_parita='dispari')
except Exception as e:
    print(f"Errore PyQSP ({e}).")
    # Qui inseriamo 4 angoli fittizi (in radianti) per costruire fisicamente il circuito 
    # nel caso l'approssimazione fallisse numericamente.
    print("Fallback: uso angoli fittizi.")
    angoli_fase_qsvt = np.array([0.15, -0.42, 0.88, -0.15])
    # SCALA DI FALLBACK <---
    scala_qsvt = 1.0
    
print(f"Angoli di fase pre-calcolati per il circuito:\n{angoli_fase_qsvt}")
print("\n")

# 3. Creiamo l'Operatore Base supremo (Il vero Block-Encoding)
def U_base():
    W()
    G_tilde()
    qml.adjoint(W)()

# PennyLane calcola la matrice dell'intero blocco
matrice_U_base = qml.matrix(U_base, wire_order=tutti_i_fili)()
operatore_U_base = qml.QubitUnitary(matrice_U_base, wires=tutti_i_fili)

# 4. Definiamo i fili proiettore (L'Ancilla LCU è sufficiente per controllare il blocco)
fili_proiettore = ancilla_lcu 

# I proiettori colpiscono lo stato |0> (dim=1 in PennyLane)
lista_proiettori = [qml.PCPhase(angle, dim=1, wires=fili_proiettore) for angle in angoli_fase_qsvt]

print("\n--- COSTRUZIONE DEL GATE P_CORSIVO ---")

# 5. Definiamo formalmente il GATE P_corsivo
def P_corsivo():
    """Applica la QSVT al vero Block-Encoding"""
    qml.QSVT(operatore_U_base, lista_proiettori)

# # (Seguono i test di compilazione, lascia tutto uguale...)

# # 3. PennyLane vuole un "Operatore" formale per la QSVT. Quindi calcoliamo la matrice 
# # esatta del nostro G_tilde, estraiamo il blocco di nostro interesse e la convertiamo in un operatore unitario nativo.
# matrice_G_tilde = qml.matrix(G_tilde, wire_order=tutti_i_fili)()

# # Assumiamo ancilla_lcu di indice 0
# # Estraiamo blocco in alto a sinistra della matrice G_tilde
# def get_G_tilde_block():   
#     # The full matrix is size 2^n x 2^n. 
#     # If ancilla is the first wire, the top-left block (ancilla=0) 
#     # is the first half of the matrix rows and columns.
#     dim = matrice_G_tilde.shape[0] // 2
#     g_tilde_block = matrice_G_tilde[0:dim, 0:dim]
    
#     return g_tilde_block 
# matrice_G_tilde_block = get_G_tilde_block()

# print("Matrice G_tilde 'ritagliata' =", np.linalg.eig(matrice_G_tilde_block)[0])
# print("\n")

# operatore_G_tilde = qml.QubitUnitary(matrice_G_tilde, wires=tutti_i_fili)
    
# # 4. Definiamo i "fili proiettore". 
# # A noi serve langolino in alto a sinistra che corrisponde allo stato in cui 
# # l'ancilla LCU, l'ancilla B e i qubit dei Dati sono tutti a ZERO. 
# # La QSVT ha bisogno di saperlo per agire sul posto giusto!

# # Definiamo i fili e lo stato (tutti zeri) dell'angolino in alto a sinistra
# fili_proiettore = ancilla_lcu + qubits_dati + ancilla_b
# stato_zero = np.zeros(len(fili_proiettore), dtype=int)

# # Creiamo una lista di Phase-Shift Controllers (uno per ogni angolo).
# # 'dim=1' significa che la fase viene applicata ESATTAMENTE e SOLO allo stato 
# # in cui i fili del proiettore sono a ZERO (il nostro angolino in alto a sinistra!).
# #lista_proiettori = [qml.PCPhase(2*angle, dim=0, wires=fili_proiettore) for angle in angoli_fase_qsvt]
# lista_proiettori = [qml.PCPhase(angle, dim=1, wires=fili_proiettore) for angle in angoli_fase_qsvt]

# print("\n--- COSTRUZIONE DEL GATE P_CORSIVO ---")

# # 5. Definiamo formalmente il GATE P_corsivo
# def P_corsivo(): #già circuito_qsvt
#     """
#     Il Gate P_corsivo: la trasformazione non lineare sui valori singolari.
#     Applica la QSVT al block-encoding G_tilde.
#     Prende: 1. Operatore Base | 2. Lista di [ProiettoreIn, ProiettoreOut] | 3. Angoli
#     """
#     qml.QSVT(operatore_G_tilde, lista_proiettori)
    
# 6. Test di Compilazione del Circuito
dev_test = qml.device("default.qubit", wires=tutti_i_fili)

@qml.qnode(dev_test)
def test_qsvt_build():
    # Prepariamo lo stato iniziale: Hadamard sugli Address per creare 
    # la sovrapposizione uniforme di tutti gli input.
    for wire in qubits_address:
        qml.Hadamard(wires=wire)
        
    # Applichiamo la scatola QSVT che incastra G_tilde + Polinomio
    P_corsivo()
    
    # Ritorniamo il vettore di stato finale
    return qml.state()

print("\n--- COMPILAZIONE CIRCUITO TEST QSVT P_CORSIVO ---")
try:
    stato_finale = test_qsvt_build()
    print("-> SUCCESSO: La compilazione del nodo QSVT su G_tilde è andata a buon fine!")
    print(f"-> Il vettore di stato risultante ha {len(stato_finale)} ampiezze complesse.")
except Exception as e:
    print(f"-> Errore nella compilazione QSVT: {e}")



#poly = np.array([0, 1.0, 0, -1/2, 0, 1/3])



--- IMPOSTAZIONE QSVT ---
Vogliamo applicare il polinomio: P(x) = 0.05x^6 + 0.05x^5 + 0.1x^4 + 0.1x^3 + 0.3x^2 + 0.2x + 0.2

--- GENERATORE DI ANGOLI QSVT ---
Modalità: Inserimento diretto del Polinomio
output qsvt:  0.11650052571416913
P(t) =        0.11650052571428572
Angoli di fase pre-calcolati per il circuito:
[-5.49778714  1.57079633  1.57079633  0.39507987  1.24490655  0.76093315]



--- COSTRUZIONE DEL GATE P_CORSIVO ---

--- COMPILAZIONE CIRCUITO TEST QSVT P_CORSIVO ---
-> SUCCESSO: La compilazione del nodo QSVT su G_tilde è andata a buon fine!
-> Il vettore di stato risultante ha 64 ampiezze complesse.


In [9]:
# ==========================================
# PARTE 7: CIRCUITO FINALE ED ESTRAZIONE (Fig. 4)
# ==========================================

print("\n--- ESECUZIONE CIRCUITO NTCA FINALE ---")

print("\n--- COSTRUZIONE DELLO STATO FINALE ---")

# 1. Costruiamo lo STATO quantistico
dev_finale = qml.device("default.qubit", wires=tutti_i_fili)

@qml.qnode(dev_finale)
def circuito_stato_finale():
    """
    H su address -> W -> P_corsivo -> W_dagger
    """
    
    # a) H sul registro 'add' (Inizializzazione della sovrapposizione)
    for wire in qubits_address:
        qml.Hadamard(wires=wire)
        
    # # b) Gate W (Allinea i dati con gli indirizzi)
    # W()
    
    # c) Gate P_corsivo (Applica il polinomio QSVT)
    P_corsivo()
    
    # # d) Gate W_dagger (Smonta l'allineamento per isolare i risultati puliti)
    # qml.adjoint(W)()
    
    # Restituisce il vettore di stato gigante da cui pescheremo i risultati
    return qml.state()

# 2. Esecuzione ed Estrazione
print("-> Esecuzione del circuito in corso...")
stato_ntca = circuito_stato_finale()
print("   Circuito eseguito! Vettore di stato ottenuto.\n")

print("--- RISULTATI ESTRATTI ---")
# lista inizializzata con 0, dove andranno i valori finali
risultati_quantistici = np.zeros(N_pad, dtype=complex)

# Il vettore ha 64 elementi. Cerchiamo la fetta in cui b_anc, data e B sono a 0.
for i, ampiezza in enumerate(stato_ntca):
    # Convertiamo l'indice in stringa binaria (es: '010110')
    binario = format(i, f'0{len(tutti_i_fili)}b')
    
    lcu_str = binario[0]                   # 'b_anc' nel disegno
    addr_str = binario[1:1+n]              # 'add' nel disegno
    dati_e_b_str = binario[1+n:]           # 'data' e 'B' nel disegno
    
    # La regola: prendi solo se 'b_anc' è 0 E tutti i 'data' e 'B' sono 0
    if lcu_str == '0' and all(bit == '0' for bit in dati_e_b_str):
        k = int(addr_str, 2) # Converte l'indirizzo da binario a decimale
        # Moltiplichiamo per sqrt(N_pad) per compensare le porte H iniziali
        risultati_quantistici[k] = ampiezza * np.sqrt(N_pad)

# Stampiamo i valori finali trovati negli address |k>
for k in range(N_pad):
    valore = risultati_quantistici[k]
    # Arrotondiamo per pulire i piccolissimi errori di calcolo del simulatore
    valore_pulito = np.round(valore.real, 4) + 1j * np.round(valore.imag, 4)
    print(f"Indirizzo |{k}> : {valore_pulito}")

print("\n")



for k in range(N_pad):
    valore = risultati_quantistici[k]

    # 1. NELLA QSVT, IL RISULTATO È SOLO NELLA PARTE REALE
    valore_reale = valore.real
    
    # 2. RIPRISTINIAMO IL VALORE ORIGINALE (Denormalizzazione)
    risultato_vero = valore_reale * scala_qsvt
    
    # 3. Stampiamo il risultato finale pulito, Arrotondiamo per pulire i piccolissimi errori di calcolo del simulatore
    print(f"Indirizzo |{k}> : {risultato_vero:.4f}")

print("\n")


--- ESECUZIONE CIRCUITO NTCA FINALE ---

--- COSTRUZIONE DELLO STATO FINALE ---
-> Esecuzione del circuito in corso...
   Circuito eseguito! Vettore di stato ottenuto.

--- RISULTATI ESTRATTI ---
Indirizzo |0> : (0.0447+0.2063j)
Indirizzo |1> : (-0.2821-0.5533j)
Indirizzo |2> : (0.791-0.3233j)
Indirizzo |3> : (0.0191+0.0898j)


Indirizzo |0> : 0.0156
Indirizzo |1> : -0.0988
Indirizzo |2> : 0.2771
Indirizzo |3> : 0.0067


